# OCR proof eval — FUNSD + SROIE (Google Colab **or** Kaggle)

**One notebook for both platforms:** `notebooks/colab/ocr_funsd_sroie_eval_colab.ipynb`

The setup cell auto-detects **Colab** vs **Kaggle** and sets paths, clone, and download/output steps accordingly.

1. **GPU runtime** (Colab: Runtime → T4 GPU; Kaggle: GPU on)
2. **Run all** in order (deps → clone → … → eval; do not skip)

**Must see before the long eval:** `Platform=...` → `Paddle GPU probe OK` → `GATE PASSED` → `PaddleOCR ready` → VRAM **delta after OCR warmup** (not static 117 MB) (not ~0.1 GB).

Full eval ~45–75 min (~1,186 pages).

| Platform | Get artifacts |
|----------|----------------|
| **Colab** | Last cell downloads `ocr_proof_bundle.zip` |
| **Kaggle** | `kaggle kernels output` → `python scripts/sync_kaggle_ocr_proof.py` |


In [ ]:
# =============================================================================
# EDIT THESE — then Run All (must run this cell before Paddle loads)
# =============================================================================
rec_batch_num = 96          # MAX rec_batch (cap 96). Use 64/32 if OOM on T4
ocr_paddle_min_side = 1280  # MAX upscale side; use 960 if OOM
ocr_paddle_use_cls = True   # Angle cls on = more GPU; False saves VRAM
ocr_force_reeval = True     # Required on cloud: False reuses stale proof → 0.0 metrics
repo_github_url = "https://github.com/leemingloon/ocr-agentic-rag.git"
repo_dir_name = "ocr-agentic-rag"
# =============================================================================

import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Kaggle kernels expose /kaggle/input; Colab does not — detect Kaggle only when not Colab
IS_KAGGLE = (not IN_COLAB) and Path("/kaggle/input").is_dir()
IS_CLOUD = IN_COLAB or IS_KAGGLE
if IS_KAGGLE:
    CLOUD_PLATFORM = "Kaggle"
elif IN_COLAB:
    CLOUD_PLATFORM = "Colab"
else:
    raise RuntimeError(
        "Run on Google Colab (T4 GPU) or Kaggle (GPU). "
        "Upload notebooks/colab/ocr_funsd_sroie_eval_colab.ipynb — one file for both."
    )

KAGGLE_REPO_DATASET = os.environ.get("KAGGLE_REPO_DATASET", "leemingloon/ocr-agentic-rag")
KAGGLE_CLONE_DIR = Path("/kaggle/working/ocr-agentic-rag")

if IS_KAGGLE:
    OUT_DIR = Path("/kaggle/working")
else:
    CONTENT = Path("/content")
    OUT_DIR = CONTENT / "ocr_colab_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["OCR_USE_GPU"] = "1"
os.environ["OCR_SKIP_TESSERACT_ENSEMBLE"] = "1"
os.environ["OCR_FAST"] = "0"
os.environ["OCR_PREFETCH"] = "1"
os.environ["OCR_REC_BATCH_NUM"] = str(rec_batch_num)
os.environ["OCR_PADDLE_MIN_SIDE"] = str(ocr_paddle_min_side)
os.environ["OCR_PADDLE_USE_CLS"] = "1" if ocr_paddle_use_cls else "0"
os.environ["OCR_FORCE_REEVAL"] = "1" if ocr_force_reeval else "0"
FORCE_REEVAL = ocr_force_reeval

def show_gpu_memory() -> None:
    if not shutil.which("nvidia-smi"):
        print("No nvidia-smi — enable GPU runtime (Colab T4 / Kaggle GPU)")
        return
    p = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free", "--format=csv,noheader,nounits"],
        capture_output=True,
        text=True,
    )
    name, total, used, free = [x.strip() for x in p.stdout.strip().split(",")]
    tg, ug, fg = float(total) / 1024, float(used) / 1024, float(free) / 1024
    print(f"{name}: {ug:.2f} / {tg:.2f} GB used ({100 * float(used) / float(total):.1f}%), {fg:.2f} GB free")

def _assert_paddle_cuda_ready(min_vram_mb: float = 0) -> None:
    import importlib.metadata as _md

    _dist = None
    for _n in ("paddlepaddle-gpu", "paddlepaddle"):
        try:
            _dist = f"{_n}=={_md.version(_n)}"
            break
        except _md.PackageNotFoundError:
            pass
    import paddle

    _cc = paddle.device.is_compiled_with_cuda()
    _nd = int(paddle.device.cuda.device_count())
    paddle.device.set_device("gpu")
    _place = str(paddle.to_tensor([1.0]).place)
    print(f"pip={_dist} cuda_compiled={_cc} devices={_nd} tensor_place={_place}")
    if _dist and _dist.startswith("paddlepaddle==") and "gpu" not in _dist:
        raise RuntimeError("CPU paddle wheel — restart runtime, Run all from top")
    if not _cc or _nd < 1 or ("gpu" not in _place.lower() and "cuda" not in _place.lower()):
        raise RuntimeError("Paddle not on GPU — restart runtime, Run all from top")
    if min_vram_mb > 0:
        raise ValueError(
            "Use Paddle cell warmup (VRAM delta after ocr()), not static nvidia-smi after download."
        )
    print("Paddle GPU probe OK")


print(f"Platform={CLOUD_PLATFORM}  IN_COLAB={IN_COLAB}  IS_KAGGLE={IS_KAGGLE}")
print(f"OUT_DIR={OUT_DIR}")
print(f"KAGGLE_REPO_DATASET={KAGGLE_REPO_DATASET}")
print(
    f"rec_batch_num={rec_batch_num}  min_side={ocr_paddle_min_side}  "
    f"force_reeval={ocr_force_reeval}  OCR_USE_GPU={os.environ.get('OCR_USE_GPU')}"
)


In [ ]:
# GPU check (run AFTER enabling GPU runtime)
!nvidia-smi

show_gpu_memory()


In [ ]:
# Dependencies — pip only (no repo clone needed). Run BEFORE clone cell.
_pre = [m for m in sys.modules if m.startswith("paddle")]
if _pre:
    raise RuntimeError(
        "Paddle already imported before GPU wheel install: "
        f"{_pre}. Runtime → Restart session → Run all from cell 1."
    )
if IS_CLOUD:
    get_ipython().run_line_magic(
        "system",
        "apt-get update -qq && apt-get install -y -qq tesseract-ocr git > /dev/null",
    )
get_ipython().run_line_magic("system", "pip uninstall -y paddlepaddle paddlepaddle-gpu -q 2>/dev/null || true")
get_ipython().run_line_magic(
    "system",
    'pip install -q "paddlepaddle-gpu==2.6.2" "paddleocr>=2.7.3,<3.0" '
    "opencv-python-headless pytesseract pillow pyarrow datasets scipy numpy pandas tqdm python-dotenv requests",
)
# Verify GPU wheel (no ocr_pipeline import — clone has not run yet)
for _m in [k for k in list(sys.modules) if k == "paddle" or k.startswith("paddle")]:
    del sys.modules[_m]
import importlib.metadata as _md

_dist = None
for _n in ("paddlepaddle-gpu", "paddlepaddle"):
    try:
        _dist = f"{_n}=={_md.version(_n)}"
        break
    except _md.PackageNotFoundError:
        pass
import paddle

if _dist and _dist.startswith("paddlepaddle==") and "gpu" not in _dist:
    raise RuntimeError("CPU paddle wheel installed — pip install paddlepaddle-gpu failed")
if not paddle.device.is_compiled_with_cuda():
    raise RuntimeError("paddle not CUDA-enabled — need paddlepaddle-gpu on GPU runtime")
paddle.device.set_device("gpu")
_place = str(paddle.to_tensor([1.0]).place)
if "gpu" not in _place.lower() and "cuda" not in _place.lower():
    raise RuntimeError(f"Paddle tensor not on GPU: {_place}")
print(f"deps OK: {_dist} place={_place} (GPU wheel — clone/patch cells come next)")

import logging
logging.getLogger("ppocr").setLevel(logging.ERROR)


In [ ]:
# Clone or locate repo (platform-specific), then apply cloud patches
from pathlib import Path
import os
import sys
import shutil
import subprocess

if "IN_COLAB" not in globals():
    raise RuntimeError(
        "Run the setup cell above first (must print Platform=Colab). "
        "Use Runtime → Run all — do not run this cell alone."
    )

def _has_eval_runner(path: Path) -> bool:
    return (path / "eval_runner.py").is_file()

def find_repo_root_kaggle() -> Path:
    candidates: list[Path] = []
    here = Path.cwd().resolve()
    for n in (
        here,
        here.parent,
        here.parent.parent,
        here.parent.parent.parent,
        here.parent.parent.parent.parent,
    ):
        if _has_eval_runner(n):
            candidates.append(n.resolve())

    input_root = Path("/kaggle/input")
    if KAGGLE_REPO_DATASET:
        for slug in (
            KAGGLE_REPO_DATASET,
            KAGGLE_REPO_DATASET.replace("/", "-"),
            KAGGLE_REPO_DATASET.split("/")[-1],
        ):
            p = input_root / slug
            if _has_eval_runner(p):
                candidates.append(p.resolve())
    if input_root.is_dir():
        for p in sorted(input_root.iterdir()):
            if not p.is_dir():
                continue
            for sub in (p, p / "ocr-agentic-rag", p / "repo"):
                if _has_eval_runner(sub):
                    candidates.append(sub.resolve())
    if _has_eval_runner(KAGGLE_CLONE_DIR):
        candidates.append(KAGGLE_CLONE_DIR.resolve())

    if not candidates:
        print(f"No repo on /kaggle/input — cloning {repo_github_url} ...", flush=True)
        if KAGGLE_CLONE_DIR.is_dir() and not _has_eval_runner(KAGGLE_CLONE_DIR):
            shutil.rmtree(KAGGLE_CLONE_DIR, ignore_errors=True)
        if not _has_eval_runner(KAGGLE_CLONE_DIR):
            proc = subprocess.run(
                ["git", "clone", "--depth", "1", repo_github_url, str(KAGGLE_CLONE_DIR)],
                capture_output=True,
                text=True,
            )
            if proc.returncode != 0:
                raise FileNotFoundError(
                    "git clone failed. Attach a Kaggle dataset with eval_runner.py, "
                    "or fix repo_github_url.\n"
                    f"stderr: {proc.stderr}\nstdout: {proc.stdout}"
                )
        if _has_eval_runner(KAGGLE_CLONE_DIR):
            candidates.append(KAGGLE_CLONE_DIR.resolve())

    if not candidates:
        raise FileNotFoundError(
            "eval_runner.py not found. Enable Internet, attach repo dataset, or fix repo_github_url."
        )
    return candidates[0]

if IS_KAGGLE:
    REPO_ROOT = find_repo_root_kaggle()
elif IN_COLAB:
    REPO_ROOT = Path("/content") / repo_dir_name
    if (REPO_ROOT / "eval_runner.py").is_file():
        print(f"Repo exists at {REPO_ROOT} — git pull")
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=False)
    else:
        if REPO_ROOT.is_dir():
            shutil.rmtree(REPO_ROOT, ignore_errors=True)
        print(f"Cloning {repo_github_url} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", repo_github_url, str(REPO_ROOT)],
            check=True,
        )
else:
    raise RuntimeError("IS_CLOUD was False — run on Colab or Kaggle only")

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"REPO_ROOT: {REPO_ROOT.resolve()}")
print(f"Platform: {CLOUD_PLATFORM}")
# Patch stale GitHub clone (slim __init__, compat shim, no anthropic on import path)
import re as _re
import base64 as _b64

# Bundled inside this notebook — works when GitHub clone lacks ocr_pipeline/compat/
_EMB_BUNDLE_B64 = '...'

def _write_notebook_cloud_bundle(root: Path) -> None:
    ns: dict = {}
    exec(_b64.b64decode(_EMB_BUNDLE_B64), ns)
    ns["write_cloud_bundle"](root)

def _bundle_compat_if_missing(root: Path) -> None:
    shim = root / "ocr_pipeline" / "compat" / "paddle_langchain_shim.py"
    if shim.is_file():
        return
    for _rel in (
        "notebooks/colab/embedded_cloud_bundle.py",
        "scripts/apply_ocr_cloud_patches.py",
    ):
        emb = root / _rel
        if not emb.is_file():
            continue
        import importlib.util
        spec = importlib.util.spec_from_file_location("_cloud_bundle", emb)
        mod = importlib.util.module_from_spec(spec)
        assert spec.loader is not None
        spec.loader.exec_module(mod)
        if hasattr(mod, "write_cloud_bundle"):
            mod.write_cloud_bundle(root)
        elif hasattr(mod, "ensure_cloud_bundle"):
            mod.ensure_cloud_bundle(root)
        if shim.is_file():
            return
    _write_notebook_cloud_bundle(root)
    if not shim.is_file():
        raise RuntimeError("Failed to write ocr_pipeline.compat (notebook bundle)")

def _apply_cloud_patches(root: Path) -> None:
    ps = root / "scripts" / "apply_ocr_cloud_patches.py"
    if ps.is_file():
        pr = subprocess.run([sys.executable, str(ps), str(root)], cwd=str(root))
        if pr.returncode == 0:
            return
        print("apply_ocr_cloud_patches failed; using inline fallback")
    init_py = root / "ocr_pipeline" / "__init__.py"
    if init_py.is_file():
        t = init_py.read_text(encoding="utf-8")
        if "from .recognition import" in t or ("HybridOCR" in t and "__getattr__" not in t):
            init_py.write_text(
                '"""Slim init for OCR eval."""\n'
                '__version__ = "1.0.0"\n__all__ = ["__version__"]\n',
                encoding="utf-8",
            )
            print("patched ocr_pipeline/__init__.py")
    vo = root / "ocr_pipeline" / "recognition" / "vision_ocr.py"
    if vo.is_file():
        t = vo.read_text(encoding="utf-8")
        if _re.search(r"^from anthropic import Anthropic", t, _re.MULTILINE):
            vo.write_text(
                _re.sub(r"^from anthropic import Anthropic\s*\n", "", t, count=1, flags=_re.M),
                encoding="utf-8",
            )
            print("patched vision_ocr.py")
    _bundle_compat_if_missing(root)
    verify = """
from pathlib import Path
assert Path("ocr_cloud_eval_entry.py").is_file()
_er = Path("eval_runner.py").read_text(encoding="utf-8")
assert "def run_ocr_all_splits" in _er or "ocr_cloud_eval_entry" in _er
from ocr_pipeline.compat.paddle_langchain_shim import install_paddle_langchain_shim
install_paddle_langchain_shim()
from ocr_pipeline.detection.paddleocr_detector import PADDLEOCR_AVAILABLE
from ocr_cloud_eval_entry import _ocr_use_ensemble, run_ocr_all_splits, warmup_ocr_pipeline
try:
    from ocr_pipeline.detection.paddleocr_detector import get_or_build_native_paddle_ocr
except ImportError:
    from ocr_pipeline.detection.paddle_cloud_api import get_or_build_native_paddle_ocr
import sys
assert "anthropic" not in sys.modules
assert callable(run_ocr_all_splits)
assert callable(get_or_build_native_paddle_ocr)
print("verify_ok", PADDLEOCR_AVAILABLE)
"""
    p = subprocess.run(
        [sys.executable, "-c", verify],
        cwd=str(root),
        capture_output=True,
        text=True,
        env={**os.environ, "PYTHONPATH": str(root)},
    )
    if p.returncode != 0:
        print(p.stdout, p.stderr)
        raise RuntimeError("Patch verify failed")

_apply_cloud_patches(REPO_ROOT)
for _k in list(sys.modules):
    if _k == "ocr_pipeline" or _k.startswith("ocr_pipeline."):
        del sys.modules[_k]
print("Cloud patches OK.")


In [ ]:
# Download FUNSD + SROIE parquet from HuggingFace (inline — no scripts/ file on GitHub clone)
_OCR_SPLITS = [
    ("FUNSD", "train", "nielsr/funsd"),
    ("FUNSD", "test", "nielsr/funsd"),
    ("SROIE", "train", "jsdnrs/ICDAR2019-SROIE"),
    ("SROIE", "test", "jsdnrs/ICDAR2019-SROIE"),
]

def _export_ocr_parquet(dataset: str, split: str, hf_repo: str) -> None:
    from datasets import load_dataset

    out_dir = REPO_ROOT / "data" / "ocr" / dataset / split
    out_dir.mkdir(parents=True, exist_ok=True)
    existing = [p for p in out_dir.glob("*.parquet") if p.stat().st_size > 1000]
    if existing:
        print(f"  {dataset}/{split}: already have {len(existing)} parquet file(s)")
        return
    print(f"  Downloading {hf_repo} split={split} ...", flush=True)
    ds = load_dataset(hf_repo, split=split)
    out_path = out_dir / f"{dataset.lower()}_{split}.parquet"
    ds.to_parquet(str(out_path))
    print(f"  wrote {out_path} ({ds.num_rows} rows, {out_path.stat().st_size // 1024} KB)", flush=True)

print("Export OCR parquet to", REPO_ROOT / "data" / "ocr")
for _ds, _sp, _hf in _OCR_SPLITS:
    try:
        _export_ocr_parquet(_ds, _sp, _hf)
    except Exception as _exc:
        raise RuntimeError(f"HF export failed for {_ds}/{_sp}: {_exc}") from _exc

for ds, split in [("FUNSD", "train"), ("FUNSD", "test"), ("SROIE", "train"), ("SROIE", "test")]:
    p = REPO_ROOT / "data" / "ocr" / ds / split
    n = sum(1 for _ in p.glob("*.parquet") if _.stat().st_size > 1000) if p.is_dir() else 0
    print(f"data/ocr/{ds}/{split}: parquet shards={n}")
    if n < 1:
        raise RuntimeError(
            f"No parquet for {ds}/{split}. Enable Internet on Colab/Kaggle and re-run this cell."
        )
print("Parquet gate OK — safe to run OCR eval")


In [ ]:
# ========== MANDATORY GATE — must print "GATE PASSED" before GPU eval ==========

def _cloud_ocr_gate(root: Path) -> None:
    root = Path(root).resolve()
    if not (root / "eval_runner.py").is_file():
        raise FileNotFoundError("Run clone cell first.")
    # Re-ensure compat (apply script on GitHub may be old)
    if "_bundle_compat_if_missing" in globals():
        _bundle_compat_if_missing(root)
    else:
        _write_notebook_cloud_bundle(root)
    _patch_script = root / "scripts" / "apply_ocr_cloud_patches.py"
    if _patch_script.is_file():
        _pr = subprocess.run([sys.executable, str(_patch_script), str(root)], cwd=str(root))
        if _pr.returncode != 0:
            print("apply_ocr_cloud_patches returned", _pr.returncode, "(continuing if compat exists)")
    for _k in list(sys.modules):
        if _k == "ocr_pipeline" or _k.startswith("ocr_pipeline."):
            del sys.modules[_k]
    _verify = """
from ocr_pipeline.compat.paddle_langchain_shim import install_paddle_langchain_shim
install_paddle_langchain_shim()
from ocr_pipeline.detection.paddleocr_detector import PADDLEOCR_AVAILABLE
from pathlib import Path
assert Path("ocr_cloud_eval_entry.py").is_file()
_er = Path("eval_runner.py").read_text(encoding="utf-8")
assert "def run_ocr_all_splits" in _er or "ocr_cloud_eval_entry" in _er
try:
    from eval_runner import run_ocr_all_splits, _ocr_use_ensemble
except ImportError:
    from ocr_cloud_eval_entry import run_ocr_all_splits, _ocr_use_ensemble
try:
    from ocr_pipeline.detection.paddleocr_detector import get_or_build_native_paddle_ocr
except ImportError:
    from ocr_pipeline.detection.paddle_cloud_api import get_or_build_native_paddle_ocr
import sys
assert "anthropic" not in sys.modules
assert callable(run_ocr_all_splits)
assert callable(_ocr_use_ensemble)
assert callable(get_or_build_native_paddle_ocr)
print("GATE_VERIFY_OK", PADDLEOCR_AVAILABLE)
"""
    _vp = subprocess.run(
        [sys.executable, "-c", _verify],
        cwd=str(root),
        capture_output=True,
        text=True,
        env={**os.environ, "PYTHONPATH": str(root)},
    )
    if _vp.returncode != 0:
        print(_vp.stdout, _vp.stderr)
        raise RuntimeError("GATE FAILED — fix before Paddle (anthropic/import path)")
    _audit = root / "scripts" / "audit_ocr_eval_cloud.py"
    if _audit.is_file():
        _ar = subprocess.run([sys.executable, str(_audit)], cwd=str(root))
        if _ar.returncode != 0:
            raise RuntimeError("audit_ocr_eval_cloud failed")
    print("=" * 60)
    print("GATE PASSED — safe to load Paddle / run_ocr_all_splits")
    print(_vp.stdout.strip())
    print("=" * 60)

_cloud_ocr_gate(REPO_ROOT)


In [ ]:
# Paddle load + GPU VRAM check (must be >> 0.1 GB before full eval)

# HARD GATE again before any ocr_pipeline import in this kernel
try:
    _cloud_ocr_gate(REPO_ROOT)
except NameError:
    raise RuntimeError("Run the GATE cell above first (must print GATE PASSED)")

from ocr_pipeline.compat.paddle_langchain_shim import install_paddle_langchain_shim
install_paddle_langchain_shim()
from ocr_pipeline.detection.paddleocr_detector import PADDLEOCR_AVAILABLE
try:
    from ocr_pipeline.detection.paddleocr_detector import get_or_build_native_paddle_ocr
except ImportError:
    from ocr_pipeline.detection.paddle_cloud_api import get_or_build_native_paddle_ocr
from ocr_pipeline.ocr_eval_config import paddle_rec_batch_num, paddle_min_side, paddle_use_angle_cls

print(f"PADDLEOCR_AVAILABLE={PADDLEOCR_AVAILABLE}")
print(f"rec_batch_num={paddle_rec_batch_num()} min_side={paddle_min_side()} angle_cls={paddle_use_angle_cls()}")
print(f"PADDLEOCR_AVAILABLE={PADDLEOCR_AVAILABLE}")
print(f"rec_batch_num={paddle_rec_batch_num()} min_side={paddle_min_side()} angle_cls={paddle_use_angle_cls()}")
if PADDLEOCR_AVAILABLE:
    print("Loading PaddleOCR (GPU ctor + OCR warmup)...")
    _paddle = get_or_build_native_paddle_ocr(show_log=False)
    try:
        from ocr_pipeline.paddle_gpu_check import warmup_paddleocr_allocates_gpu
        warmup_paddleocr_allocates_gpu(_paddle, min_used_mb=200.0, min_delta_mb=50.0)
    except ImportError:
        import numpy as np
        def _vram_mb() -> float:
            p = subprocess.run(
                ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                capture_output=True,
                text=True,
            )
            return float(p.stdout.strip().splitlines()[0])
        _b = _vram_mb()
        _img = np.full((640, 640, 3), 128, dtype=np.uint8)
        for _kw in ({"cls": False}, {}):
            try:
                _paddle.ocr(_img, **_kw)
                break
            except TypeError:
                _paddle.ocr(_img)
                break
        _a = _vram_mb()
        print(f"[Paddle] VRAM before={_b:.0f} after={_a:.0f} delta={_a - _b:.0f} MB (inline warmup)")
        if (_a - _b) < 50 and _a < 200:
            raise RuntimeError("Inline warmup: VRAM did not rise — Paddle likely on CPU")
    print("PaddleOCR ready — GPU warmup passed.")
else:
    raise RuntimeError("PaddleOCR not available")
show_gpu_memory()
_assert_paddle_cuda_ready(min_vram_mb=0)
_post_env = {
    **os.environ,
    "OCR_USE_GPU": "1",
    "OCR_SKIP_TESSERACT_ENSEMBLE": os.environ.get("OCR_SKIP_TESSERACT_ENSEMBLE", "1"),
    "OCR_FAST": os.environ.get("OCR_FAST", "1"),
    "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK": "True",
}
_POST_PADDLE_INLINE = """
from ocr_pipeline.compat.paddle_langchain_shim import install_paddle_langchain_shim
install_paddle_langchain_shim()
try:
    from eval_runner import run_ocr_all_splits, warmup_ocr_pipeline, _ocr_use_ensemble
except ImportError:
    from eval_runner import run_ocr_all_splits, warmup_ocr_pipeline
    from ocr_cloud_eval_entry import _ocr_use_ensemble
from ocr_pipeline.recognition.hybrid_ocr import HybridOCR
HybridOCR(use_detection_router=False, use_vision_augmentation=False, use_ensemble_for_accuracy=False)
import sys
assert "anthropic" not in sys.modules
assert not _ocr_use_ensemble("FUNSD")
assert callable(run_ocr_all_splits)
print("POST_PADDLE_VERIFY_OK")
"""
_audit = REPO_ROOT / "scripts" / "audit_ocr_eval_cloud.py"
if _audit.is_file():
    _r = subprocess.run(
        [sys.executable, str(_audit), "--smoke-paddle"],
        cwd=str(REPO_ROOT),
        env=_post_env,
        capture_output=True,
        text=True,
        check=False,
    )
else:
    print("Note: scripts/audit_ocr_eval_cloud.py not on GitHub clone — inline Post-Paddle verify")
    _r = subprocess.run(
        [sys.executable, "-c", _POST_PADDLE_INLINE],
        cwd=str(REPO_ROOT),
        env={**_post_env, "PYTHONPATH": str(REPO_ROOT)},
        capture_output=True,
        text=True,
        check=False,
    )
if _r.stdout:
    print(_r.stdout)
if _r.stderr:
    print(_r.stderr, file=sys.stderr)
if _r.returncode != 0:
    raise RuntimeError(
        "Post-Paddle audit failed — do not start run_ocr_all_splits. "
        "See output above (re-run clone cell; need ocr_cloud_eval_entry + eval_runner stub)."
    )
print("Post-Paddle audit OK.")

_assert_paddle_cuda_ready(min_vram_mb=0)


In [ ]:
import time
from eval_runner import run_ocr_all_splits

log_path = OUT_DIR / "ocr_eval_run.log"
t0 = time.perf_counter()
# VRAM gate already ran in Paddle cell (warmup after ocr())
print(f"Starting run_ocr_all_splits (force_reeval={ocr_force_reeval}) ...", flush=True)
print("Expect ~45–75 min on T4 GPU. If ~4 s/page and VRAM ~0.1 GB, STOP — Paddle is on CPU.", flush=True)
with open(log_path, "a", encoding="utf-8") as logf:
    logf.write(f"\n=== run_ocr_all_splits force_reeval={ocr_force_reeval} ===\n")
run_ocr_all_splits(force_reeval=ocr_force_reeval, proof_dir=str(REPO_ROOT / "data" / "proof"))
elapsed = time.perf_counter() - t0
print(f"All splits finished in {elapsed / 60:.1f} min", flush=True)
if elapsed < 600:
    raise RuntimeError(
        f"Eval finished in {elapsed / 60:.1f} min — expected ~45+ min. "
        "Set ocr_force_reeval=True and confirm parquet shards>=1 above."
    )
show_gpu_memory()


In [ ]:
PROOF_SRC = REPO_ROOT / "data" / "proof" / "ocr"
BUNDLE = OUT_DIR / "ocr_proof_bundle"
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
shutil.copytree(PROOF_SRC, BUNDLE)

summary = {
    "environment": CLOUD_PLATFORM,
    "in_colab": IN_COLAB,
    "is_kaggle": IS_KAGGLE,
    "repo_root": str(REPO_ROOT),
    "splits": {},
}
for dataset in ("funsd", "sroie"):
    for split in ("train", "test"):
        avg = PROOF_SRC / dataset / split / f"{dataset}_{split}_avg.json"
        if avg.is_file():
            summary["splits"][f"{dataset}_{split}"] = json.loads(avg.read_text(encoding="utf-8"))

summary_path = OUT_DIR / "ocr_eval_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
shutil.copy2(summary_path, BUNDLE / "ocr_eval_summary.json")
if log_path.is_file():
    _log_dst = BUNDLE / "ocr_eval_run.log"
    if log_path.resolve() != _log_dst.resolve():
        shutil.copy2(log_path, _log_dst)

zip_base = str(OUT_DIR / "ocr_proof_bundle")
zip_path = Path(shutil.make_archive(zip_base, "zip", OUT_DIR, "ocr_proof_bundle"))
if not zip_path.is_file():
    raise FileNotFoundError(f"Zip not created: {zip_path}")

print("=" * 60)
print("SPLIT METRICS")
for k, v in summary.get("splits", {}).items():
    keys = [x for x in v if x.endswith("_mean") or x == "sample_count"]
    print(k, {m: v.get(m) for m in keys if m in v})
_wr = summary.get("splits", {}).get("funsd_train", {}).get("word_recall_mean", 0)
if _wr == 0 and summary.get("splits", {}).get("funsd_train", {}).get("sample_count", 0) > 0:
    raise RuntimeError("All metrics are 0 — keep ocr_force_reeval=True and re-run eval")
print("=" * 60)
print(f"Zip ready: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
print(f"Summary:   {summary_path}")
print("Local sync: OCR_PROOF_SRC=<unzip-dir> python scripts/sync_kaggle_ocr_proof.py")

if IN_COLAB:
    from google.colab import files

    print("Starting browser download of ocr_proof_bundle.zip ...")
    files.download(str(zip_path))
    files.download(str(summary_path))
    print("DOWNLOAD DONE — check downloads folder for ocr_proof_bundle.zip")
elif IS_KAGGLE:
    print("=" * 60)
    print("Kaggle: pull kernel outputs on your machine:")
    print("  kaggle kernels output leemingloon/ocr-funsd-sroie-eval-kaggle \\")
    print("    -p notebooks/kaggle-kernels/ocr_funsd_sroie_eval_kaggle/kaggle_outputs")
    print("  python scripts/sync_kaggle_ocr_proof.py")
    print(f"Bundle dir: {BUNDLE}")
else:
    raise RuntimeError("Not on Colab or Kaggle — cannot deliver artifacts")


## After the kernel finishes (Kaggle only)

On your machine:

```bash
kaggle kernels output leemingloon/ocr-funsd-sroie-eval-kaggle \
  -p notebooks/kaggle-kernels/ocr_funsd_sroie_eval_kaggle/kaggle_outputs
python scripts/sync_kaggle_ocr_proof.py
```
